# Scoring logos with TRIBE v2

[TRIBE v2](https://huggingface.co/facebook/tribev2) predicts fMRI brain responses to naturalistic stimuli (video/audio/text). Here we treat each logo as a silent visual stimulus, get TRIBE v2's predicted brain-response vector for it, then use PCA to place all logos in a 2D space to see how they relate to each other.

In [1]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(".env")
import os

if os.environ.get("HUGGING_FACE_TOKEN"):
    login(token=os.environ["HUGGING_FACE_TOKEN"])

LOGOS_DIR = Path("logos")
CACHE_FOLDER = Path("./cache")
logo_paths = sorted(LOGOS_DIR.glob("*.png")) + sorted(LOGOS_DIR.glob("*.jpeg"))
logo_paths

[PosixPath('logos/anthropic.png'),
 PosixPath('logos/carulla.png'),
 PosixPath('logos/exito.png'),
 PosixPath('logos/truora.png'),
 PosixPath('logos/images.jpeg'),
 PosixPath('logos/openai.jpeg')]

## Load TRIBE v2

Downloads the checkpoint from Hugging Face on first run (~1GB).

In [ ]:
from tribev2.demo_utils import TribeModel

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={
        "data.text_feature.device": "cpu",
        "data.audio_feature.device": "cpu",
        "data.image_feature.image.device": "cpu",
        "data.video_feature.image.device": "cpu",
        "data.num_workers": 0,
    },
)

## Turn each logo into a short silent clip

TRIBE v2 only accepts video/audio/text files, not static images, so each logo is rendered as a short looping clip (no audio track).

In [ ]:
from moviepy import ImageClip

CLIP_DURATION = 1.0  # a static logo has no motion, so a short clip is enough
video_paths = {}
for logo_path in logo_paths:
    video_path = CACHE_FOLDER / f"{logo_path.stem}.mp4"
    clip = ImageClip(str(logo_path), duration=CLIP_DURATION).resized(height=256)
    clip.write_videofile(str(video_path), codec="libx264", audio=False, fps=8, logger=None)
    video_paths[logo_path.stem] = video_path
video_paths

## Score each logo

For each clip we build the events dataframe, run `model.predict`, and average the predicted brain response over time to get one embedding vector per logo.

In [4]:
embeddings = {}
for name, video_path in video_paths.items():
    print(f"Scoring {name}...")
    events = model.get_events_dataframe(video_path=video_path)
    preds, segments = model.predict(events=events, verbose=False)
    embeddings[name] = preds.mean(axis=0)

names = list(embeddings.keys())
X = np.stack([embeddings[name] for name in names])
X.shape

Scoring anthropic...


Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 35.42it/s]
Extracting words from audio: 0it [00:00, ?it/s]
No transcripts found, skipping
2026-08-22 00:47:38 - INFO - neuralset.events.transforms.text:56 - No Word events found, skipping
2026-08-22 00:47:38 - INFO - neuralset.events.transforms.text:175 - No Word events found, skipping
Add context to words: 0it [00:00, ?it/s]
[00:47:38 WARNING] Removing extractor audio as there are no corresponding events
[00:47:38 WARNING] Removing extractor text as there are no corresponding events
[00:47:38 INFO] Preparing extractor: video


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

video_preprocessor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

## PCA to 2D and plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(X)
name_to_path = {p.stem: p for p in logo_paths}

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=0)
for (x, y), name in zip(coords, names):
    img = Image.open(name_to_path[name]).convert("RGBA")
    ab = AnnotationBbox(OffsetImage(img, zoom=0.15), (x, y), frameon=False)
    ax.add_artist(ab)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Logos placed by TRIBE v2 predicted brain response (PCA)")
plt.show()